# 10 · Calibrated versions and the two ensembles

**Calibration.** A model's scores rank women well but may be too confident or too cautious as
chances. Each model gets two recalibrated versions: **isotonic** (a flexible step map) and
**Platt** (a logistic map, which keeps the ranking). To avoid scoring a calibrator on the data it
was fitted to, calibration is **nested**: inside each development fold, the model is fitted on
part of the fold's training women and the calibrator on the rest; the held-out women get the
calibrator fitted on the model's out-of-fold predictions.

**Stacked ensemble.** A logistic regression that blends the models' out-of-fold predictions.

**Cumulative-link ensemble.** The outcome is a threshold on a count of contacts. The best model
(by development AP) is refitted at several contact thresholds (1, 2, 3, 4, 5, 6, 8, 10), the
curves are forced to fall as the threshold rises, and they are pooled with more weight near
the target, then mapped back to a chance with an isotonic fit on the development women.

The ensembles here are built from the three core models. The explainable boosting track builds
the same two from all four.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

MODELS = ["Logistic regression", "Random Forest", "XGBoost"]

In [ ]:
for s in lib.SETS:
    for r in lib.RUNGS:
        for m in MODELS:
            for method in lib.CALIBRATIONS:
                job = lib.Job("calib", s, r, lib.calibrated_name(m, method), m)
                if not lib.is_done(P.results, job):
                    lib.run(job, P.results)
for job in lib.lineup_jobs("three"):
    if not lib.is_done(P.results, job):
        lib.run(job, P.results)

In [ ]:
cal = [lib.calibrated_name(m, c) for m in MODELS for c in lib.CALIBRATIONS]
repro.compare_cells(P.results, cal)
res = repro.compare_cells(P.results, [lib.ensemble_name("stack", "three"), lib.ensemble_name("cumlink", "three")])
print(res[["set", "rung", "model", "held_AP"]].round(3).to_string(index=False))
repro.report()